In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Reddit ETL — Analytics Demo\n",
        "\n",
        "This notebook connects to the curated Redshift data and produces\n",
        "portfolio-ready visualizations.\n",
        "\n",
        "**Prerequisites:**\n",
        "- Redshift cluster running (via Terraform or local)\n",
        "- Pipeline has run at least once and loaded data"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "import os\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "from sqlalchemy import create_engine\n",
        "\n",
        "# Connection string (adjust host as needed)\n",
        "REDSHIFT_HOST = os.getenv('REDSHIFT_HOST', 'localhost')\n",
        "REDSHIFT_DB = os.getenv('REDSHIFT_DB', 'dev')\n",
        "REDSHIFT_USER = os.getenv('REDSHIFT_USER', 'awsuser')\n",
        "REDSHIFT_PASSWORD = os.getenv('REDSHIFT_PASSWORD', 'admin')\n",
        "\n",
        "conn_str = f\"postgresql://{REDSHIFT_USER}:{REDSHIFT_PASSWORD}@{REDSHIFT_HOST}:5439/{REDSHIFT_DB}\"\n",
        "engine = create_engine(conn_str)\n",
        "\n",
        "# Load data\n",
        "df = pd.read_sql(\"SELECT * FROM reddit_posts LIMIT 1000\", engine)\n",
        "df.head()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1. Post Volume by Subreddit\n",
        "\n",
        "Simple bar chart showing which community is more active."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "plt.figure(figsize=(8, 5))\n",
        "df['subreddit'].value_counts().plot(kind='bar', color=['#FF4500', '#1a1a1a'])\n",
        "plt.title('Posts by Subreddit')\n",
        "plt.xlabel('Subreddit')\n",
        "plt.ylabel('Count')\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2. Score vs Comments (Engagement Analysis)\n",
        "\n",
        "Scatter plot reveals the relationship between upvotes and discussion volume."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "plt.figure(figsize=(10, 6))\n",
        "for sub in df['subreddit'].unique():\n",
        "    subset = df[df['subreddit'] == sub]\n",
        "    plt.scatter(subset['num_comments'], subset['score'], label=sub, alpha=0.6)\n",
        "plt.xlabel('Number of Comments')\n",
        "plt.ylabel('Score (Upvotes)')\n",
        "plt.title('Score vs Comments by Subreddit')\n",
        "plt.legend()\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3. High-Engagement Posts\n",
        "\n",
        "Posts where engagement_ratio > 10 (many upvotes relative to comments)."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "high_eng = df[df['high_engagement'] == True]\n",
        "print(f\"High-engagement posts: {len(high_eng)} / {len(df)}\")\n",
        "high_eng[['title', 'subreddit', 'score', 'num_comments', 'engagement_ratio']].head(10)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
           "source": [
        "## 4. Word Frequency in Titles\n",
        "\n",
        "Quick word cloud from cleaned titles."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "source": [
        "from collections import Counter\n",
        "import re\n",
        "\n",
        "all_words = []\n",
        "for title in df['title'].dropna():\n",
        "    words = re.findall(r'\\b[a-z]{3,}\\b', title.lower())\n",
        "    all_words.extend(words)\n",
        "\n",
        "common = Counter(all_words).most_common(15)\n",
        "words, counts = zip(*common)\n",
        "\n",
        "plt.figure(figsize=(10, 5))\n",
        "plt.barh(words[::-1], counts[::-1], color='#FF4500')\n",
        "plt.title('Top 15 Words in Post Titles')\n",
        "plt.xlabel('Frequency')\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.11.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}